<h2>Setup</h2>

In [1]:
import time

from pyvo.dal import TAPService

In [2]:
svc = TAPService("http://35.239.90.76/tap")

In [3]:
def query(sql, print_info=True):
    """Run a query, return the results as a DataFrame, get basic performance timing, and 
    print along with row count."""
    if print_info:
        print(f"Executing query: {sql}")
    start = time.perf_counter()
    res = svc.run_async(sql).resultstable
    end = time.perf_counter()
    df = res.to_table().to_pandas()
    if print_info:
        print(f"Query took {end - start:.3f} seconds and returned {len(df)} rows")
    return df

<h2>TAP Schema</h2>

In [ ]:
query("SELECT * FROM tap_schema.columns WHERE table_name LIKE 'ppdb_lsstcam.%'")

<h2>ID</h2>

In [4]:
query("SELECT * FROM ppdb_lsstcam.DiaObject WHERE diaObjectId=169760231406961662")

Executing query: SELECT * FROM ppdb_lsstcam.DiaObject WHERE diaObjectId=169760231406961662
Query took 3.179 seconds and returned 1 rows


,raErr,diaObjectId,validityStartMjdTai,validityEndMjdTai,ra,dec,decErr,u_psfFluxNdata,g_psfFluxNdata,r_psfFluxNdata,...,i_psfFluxMaxSlope,i_psfFluxErrMean,z_psfFluxMin,z_psfFluxMax,z_psfFluxMaxSlope,z_psfFluxErrMean,y_psfFluxMin,y_psfFluxMax,y_psfFluxMaxSlope,y_psfFluxErrMean
0,0.000032,169760231406961662,61029.350145,NaN,224.394815,-41.726785,0.000043,0,0,0,...,NaN,NaN,7422.667969,7422.667969,NaN,1156.7052,NaN,NaN,NaN,NaN


<h2>Cone Search</h2>

Sample ra and dec

In [ ]:
ra = 224.0
dec = -41.7

In [ ]:
query(f"""
SELECT diaObjectId, ra, dec
FROM ppdb_lsstcam.DiaObject
WHERE CONTAINS(
POINT('ICRS', ra, dec),
CIRCLE('ICRS', {ra}, {dec}, 1.0)) = 1
""")

<h2>Nearest Neighbor Search</h2>

In [ ]:
query(f"""
SELECT 
    o1.ra as ra1,
    o1.dec as dec1,
    o2.ra as ra2,
    o2.dec as dec2,
    o1.diaObjectId AS id1,
    o2.diaObjectId AS id2,
    DISTANCE(POINT('ICRS', o1.ra, o1.dec), POINT('ICRS', o2.ra, o2.dec)) AS dist
FROM ppdb_lsstcam.DiaObject AS o1
JOIN ppdb_lsstcam.DiaObject AS o2
  ON o1.diaObjectId <> o2.diaObjectId
 AND o1.validityStartMjdTai = o2.validityStartMjdTai
WHERE CONTAINS(POINT('ICRS', o1.ra, o1.dec),
               CIRCLE('ICRS', {ra}, {dec}, 0.5)) = 1
  AND DISTANCE(POINT('ICRS', o1.ra, o1.dec),
               POINT('ICRS', o2.ra, o2.dec)) < 0.02;
""")

<h2>Join</h2>

In [ ]:
query("""SELECT * FROM ppdb_lsstcam.DiaSource ds 
LEFT JOIN ppdb_lsstcam.DiaObject dob ON dob.diaObjectId = ds.diaObjectId
WHERE dob.diaObjectId=169760231406961662""")

<h2>Table scan</h2>

In [ ]:
query("SELECT * FROM ppdb.DiaObject 
WHERE r_psfFluxMean BETWEEN 1090.0 and 1100.0")

<h2>DiaObject Counts by Day</h2>

In [ ]:
query("""SELECT FLOOR(validityStartMjdTai) as mjd_tai_day, COUNT(*) as dia_objects
FROM ppdb_lsstcam.DiaObject
GROUP BY mjd_tai_day
ORDER BY mjd_tai_day DESC
""")